In [0]:
import requests
import json
from datetime import datetime, timedelta
import pandas as pd
from pyspark.sql import Row
from pyspark.sql.functions import col, lit, current_timestamp

# ============================================================================
# WEATHER API CONFIGURATION
# ============================================================================

# You'll need to get a free API key from: https://openweathermap.org/api
# Sign up and get your API key (free tier allows 1000 calls/day)
API_KEY = "YOUR_API_KEY_HERE"  # Replace with your actual API key

# Alternative: Use Open-Meteo (no API key required)
USE_OPEN_METEO = True  # Set to False to use OpenWeatherMap


# ============================================================================
# FUNCTION 1: Fetch Current Weather
# ============================================================================

def get_current_weather(location, api_key=API_KEY):
    """
    Fetch current weather for a given location
    
    Parameters:
    -----------
    location : str
        City name (e.g., "Bhopal", "Mumbai,IN", "London,UK")
    api_key : str
        OpenWeatherMap API key
    
    Returns:
    --------
    dict : Current weather data
    """
    
    if USE_OPEN_METEO:
        # Use Open-Meteo (no API key needed)
        return get_weather_open_meteo(location)
    
    url = f"http://api.openweathermap.org/data/2.5/weather"
    params = {
        'q': location,
        'appid': api_key,
        'units': 'metric'  # Celsius
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            
            weather_info = {
                'location': data['name'],
                'country': data['sys']['country'],
                'latitude': data['coord']['lat'],
                'longitude': data['coord']['lon'],
                'timestamp': datetime.fromtimestamp(data['dt']).strftime('%Y-%m-%d %H:%M:%S'),
                'temperature_celsius': data['main']['temp'],
                'feels_like_celsius': data['main']['feels_like'],
                'temp_min': data['main']['temp_min'],
                'temp_max': data['main']['temp_max'],
                'pressure_hpa': data['main']['pressure'],
                'humidity_percent': data['main']['humidity'],
                'weather_main': data['weather'][0]['main'],
                'weather_description': data['weather'][0]['description'],
                'clouds_percent': data['clouds']['all'],
                'wind_speed_mps': data['wind']['speed'],
                'wind_direction_deg': data['wind'].get('deg', None),
                'visibility_meters': data.get('visibility', None),
                'sunrise': datetime.fromtimestamp(data['sys']['sunrise']).strftime('%H:%M:%S'),
                'sunset': datetime.fromtimestamp(data['sys']['sunset']).strftime('%H:%M:%S')
            }
            
            return weather_info
        else:
            print(f"❌ API Error: {response.status_code}")
            print(f"Message: {response.text}")
            return None
            
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return None


# ============================================================================
# FUNCTION 2: Fetch Weather Forecast (5 days, 3-hour intervals)
# ============================================================================

def get_weather_forecast(location, target_datetime=None, api_key=API_KEY):
    """
    Fetch weather forecast for a given location and datetime
    
    Parameters:
    -----------
    location : str
        City name (e.g., "Bhopal", "Delhi,IN")
    target_datetime : str, optional
        Target date and time in format "YYYY-MM-DD HH:MM" (within next 5 days)
        If None, returns full 5-day forecast
    api_key : str
        OpenWeatherMap API key
    
    Returns:
    --------
    list of dict : Forecast data (up to 40 entries, 3-hour intervals)
    """
    
    if USE_OPEN_METEO:
        return get_forecast_open_meteo(location, target_datetime)
    
    url = f"http://api.openweathermap.org/data/2.5/forecast"
    params = {
        'q': location,
        'appid': api_key,
        'units': 'metric'
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            
            forecast_list = []
            
            for item in data['list']:
                forecast_info = {
                    'location': data['city']['name'],
                    'country': data['city']['country'],
                    'latitude': data['city']['coord']['lat'],
                    'longitude': data['city']['coord']['lon'],
                    'forecast_time': item['dt_txt'],
                    'temperature_celsius': item['main']['temp'],
                    'feels_like_celsius': item['main']['feels_like'],
                    'temp_min': item['main']['temp_min'],
                    'temp_max': item['main']['temp_max'],
                    'pressure_hpa': item['main']['pressure'],
                    'humidity_percent': item['main']['humidity'],
                    'weather_main': item['weather'][0]['main'],
                    'weather_description': item['weather'][0]['description'],
                    'clouds_percent': item['clouds']['all'],
                    'wind_speed_mps': item['wind']['speed'],
                    'wind_direction_deg': item['wind'].get('deg', None),
                    'visibility_meters': item.get('visibility', None),
                    'precipitation_probability': item.get('pop', 0) * 100  # Probability of precipitation
                }
                
                # Filter by target datetime if provided
                if target_datetime:
                    target_dt = datetime.strptime(target_datetime, "%Y-%m-%d %H:%M")
                    forecast_dt = datetime.strptime(item['dt_txt'], "%Y-%m-%d %H:%M:%S")
                    
                    # Find closest match (within 3 hours)
                    if abs((forecast_dt - target_dt).total_seconds()) <= 3 * 3600:
                        forecast_list.append(forecast_info)
                else:
                    forecast_list.append(forecast_info)
            
            return forecast_list
        else:
            print(f"❌ API Error: {response.status_code}")
            return None
            
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return None


# ============================================================================
# FUNCTION 3: Open-Meteo API (No API Key Required)
# ============================================================================

def get_coordinates(location):
    """Get coordinates for a location using geocoding"""
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {'name': location, 'count': 1}
    
    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get('results'):
                result = data['results'][0]
                return {
                    'name': result['name'],
                    'country': result.get('country', 'N/A'),
                    'latitude': result['latitude'],
                    'longitude': result['longitude']
                }
    except:
        pass
    return None


def get_weather_open_meteo(location):
    """Fetch current weather using Open-Meteo API (no key required)"""
    
    # Get coordinates
    coord = get_coordinates(location)
    if not coord:
        print(f"❌ Location '{location}' not found")
        return None
    
    # Fetch weather
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        'latitude': coord['latitude'],
        'longitude': coord['longitude'],
        'current': 'temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,weather_code,cloud_cover,wind_speed_10m,wind_direction_10m',
        'timezone': 'auto'
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            current = data['current']
            
            weather_info = {
                'location': coord['name'],
                'country': coord['country'],
                'latitude': coord['latitude'],
                'longitude': coord['longitude'],
                'timestamp': current['time'],
                'temperature_celsius': current['temperature_2m'],
                'feels_like_celsius': current['apparent_temperature'],
                'humidity_percent': current['relative_humidity_2m'],
                'precipitation_mm': current['precipitation'],
                'weather_code': current['weather_code'],
                'clouds_percent': current['cloud_cover'],
                'wind_speed_kmh': current['wind_speed_10m'],
                'wind_direction_deg': current['wind_direction_10m']
            }
            
            return weather_info
    except Exception as e:
        print(f"❌ Error: {str(e)}")
    return None


def get_forecast_open_meteo(location, target_datetime=None):
    """Fetch weather forecast using Open-Meteo API"""
    
    coord = get_coordinates(location)
    if not coord:
        return None
    
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        'latitude': coord['latitude'],
        'longitude': coord['longitude'],
        'hourly': 'temperature_2m,relative_humidity_2m,apparent_temperature,precipitation_probability,weather_code,cloud_cover,wind_speed_10m',
        'forecast_days': 7,
        'timezone': 'auto'
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            hourly = data['hourly']
            
            forecast_list = []
            for i in range(len(hourly['time'])):
                forecast_info = {
                    'location': coord['name'],
                    'country': coord['country'],
                    'latitude': coord['latitude'],
                    'longitude': coord['longitude'],
                    'forecast_time': hourly['time'][i],
                    'temperature_celsius': hourly['temperature_2m'][i],
                    'feels_like_celsius': hourly['apparent_temperature'][i],
                    'humidity_percent': hourly['relative_humidity_2m'][i],
                    'precipitation_probability': hourly['precipitation_probability'][i],
                    'weather_code': hourly['weather_code'][i],
                    'clouds_percent': hourly['cloud_cover'][i],
                    'wind_speed_kmh': hourly['wind_speed_10m'][i]
                }
                
                if target_datetime:
                    target_dt = datetime.strptime(target_datetime, "%Y-%m-%d %H:%M")
                    forecast_dt = datetime.fromisoformat(hourly['time'][i])
                    if abs((forecast_dt - target_dt).total_seconds()) <= 3600:
                        forecast_list.append(forecast_info)
                else:
                    forecast_list.append(forecast_info)
            
            return forecast_list
    except Exception as e:
        print(f"❌ Error: {str(e)}")
    return None


# ============================================================================
# FUNCTION 4: Create Spark DataFrame from Weather Data
# ============================================================================

def weather_to_dataframe(weather_data):
    """
    Convert weather data (single record or list) to Spark DataFrame
    
    Parameters:
    -----------
    weather_data : dict or list of dict
        Weather data from API
    
    Returns:
    --------
    DataFrame : Spark DataFrame with weather data
    """
    
    if not weather_data:
        return None
    
    # Handle single record
    if isinstance(weather_data, dict):
        weather_data = [weather_data]
    
    # Create Spark DataFrame
    df = spark.createDataFrame(weather_data)
    return df


# ============================================================================
# DEMO & USAGE EXAMPLES
# ============================================================================

print("🌤️ WEATHER DATA SCRAPER")
print("="*70)
print(f"\n✅ Functions defined:")
print("  • get_current_weather(location)")
print("  • get_weather_forecast(location, target_datetime=None)")
print("  • weather_to_dataframe(weather_data)")
print(f"\n📡 Using: {'Open-Meteo (free, no API key)' if USE_OPEN_METEO else 'OpenWeatherMap'}")
print("\n💡 Usage Examples:")
print("-" * 70)
print("\n1️⃣ Get current weather:")
print("   weather = get_current_weather('Bhopal')")
print("   df = weather_to_dataframe(weather)")
print("\n2️⃣ Get forecast for specific date/time:")
print("   forecast = get_weather_forecast('Delhi', '2026-04-20 15:00')")
print("   df = weather_to_dataframe(forecast)")
print("\n3️⃣ Get full 7-day forecast:")
print("   forecast = get_weather_forecast('Mumbai')")
print("   df = weather_to_dataframe(forecast)")
print("\n" + "="*70)

In [0]:
# ============================================================================
# WEATHER CATEGORY PROCESSOR
# Simplify weather data to 4 categories: Clear, Cloudy, Rainy, Foggy
# ============================================================================

def categorize_weather(weather_code, cloud_cover_percent=None):
    """
    Categorize weather into 4 simple outputs: Clear, Cloudy, Rainy, Foggy
    
    Parameters:
    -----------
    weather_code : int
        WMO Weather interpretation code from Open-Meteo API
    cloud_cover_percent : int, optional
        Cloud cover percentage (0-100) for additional validation
    
    Returns:
    --------
    str : One of 'Clear', 'Cloudy', 'Rainy', 'Foggy'
    
    WMO Weather Code Mapping:
    -------------------------
    0: Clear sky
    1, 2, 3: Mainly clear, partly cloudy, overcast
    45, 48: Fog and depositing rime fog
    51-57: Drizzle (light, moderate, dense)
    61-67: Rain (light, moderate, heavy)
    71-77: Snow
    80-82: Rain showers (light, moderate, violent)
    85-86: Snow showers
    95-99: Thunderstorms
    """
    
    # Fog codes
    if weather_code in [45, 48]:
        return 'Foggy'
    
    # Rain codes (drizzle, rain, rain showers)
    if weather_code in list(range(51, 68)) + list(range(80, 83)):
        return 'Rainy'
    
    # Thunderstorm codes (with or without hail)
    if weather_code in list(range(95, 100)):
        return 'Rainy'
    
    # Clear sky
    if weather_code == 0:
        # Double check with cloud cover if available
        if cloud_cover_percent is not None:
            if cloud_cover_percent <= 20:
                return 'Clear'
            else:
                return 'Cloudy'
        return 'Clear'
    
    # Partly cloudy to overcast (1, 2, 3)
    if weather_code in [1, 2, 3]:
        # Use cloud cover for more precision
        if cloud_cover_percent is not None:
            if cloud_cover_percent <= 30:
                return 'Clear'
            else:
                return 'Cloudy'
        # Default based on code
        if weather_code == 1:  # Mainly clear
            return 'Clear'
        else:  # Partly cloudy or overcast
            return 'Cloudy'
    
    # All other conditions (snow, etc.) = Cloudy
    return 'Cloudy'


def process_weather_with_category(weather_data):
    """
    Add weather category to weather data (single record or list)
    
    Parameters:
    -----------
    weather_data : dict or list of dict
        Weather data from API (current or forecast)
    
    Returns:
    --------
    dict or list of dict : Weather data with added 'weather_category' field
    """
    
    if not weather_data:
        return None
    
    # Handle single record
    if isinstance(weather_data, dict):
        weather_data = [weather_data]
        single_record = True
    else:
        single_record = False
    
    # Process each record
    processed_data = []
    for record in weather_data:
        # Create a copy to avoid modifying original
        processed_record = record.copy()
        
        # Get weather code and cloud cover
        weather_code = record.get('weather_code')
        cloud_cover = record.get('clouds_percent')
        
        # Categorize
        if weather_code is not None:
            category = categorize_weather(weather_code, cloud_cover)
            processed_record['weather_category'] = category
        else:
            # Fallback if no weather code (shouldn't happen with Open-Meteo)
            processed_record['weather_category'] = 'Unknown'
        
        processed_data.append(processed_record)
    
    # Return single record if input was single
    if single_record:
        return processed_data[0]
    else:
        return processed_data


def get_weather_simple(location, target_datetime=None):
    """
    Simplified weather function that returns only location, time, temp, and category
    
    Parameters:
    -----------
    location : str
        City name (e.g., "Bhopal", "Delhi")
    target_datetime : str, optional
        Target date and time in format "YYYY-MM-DD HH:MM" (within next 7 days)
        If None, returns current weather
    
    Returns:
    --------
    dict or list : Simplified weather data with 4-category classification
    """
    
    if target_datetime:
        # Get forecast
        data = get_weather_forecast(location, target_datetime)
    else:
        # Get current weather
        data = get_current_weather(location)
    
    if not data:
        return None
    
    # Add categories
    data_with_category = process_weather_with_category(data)
    
    # Simplify output
    if isinstance(data_with_category, dict):
        # Single record
        simple_output = {
            'location': data_with_category['location'],
            'timestamp': data_with_category.get('timestamp') or data_with_category.get('forecast_time'),
            'temperature_celsius': data_with_category['temperature_celsius'],
            'weather_category': data_with_category['weather_category']
        }
        return simple_output
    else:
        # Multiple records
        simple_output = []
        for record in data_with_category:
            simple_record = {
                'location': record['location'],
                'timestamp': record.get('timestamp') or record.get('forecast_time'),
                'temperature_celsius': record['temperature_celsius'],
                'weather_category': record['weather_category']
            }
            simple_output.append(simple_record)
        return simple_output


print("✅ WEATHER CATEGORY PROCESSOR LOADED")
print("="*70)
print("\n📊 Functions defined:")
print("  • categorize_weather(weather_code, cloud_cover_percent)")
print("  • process_weather_with_category(weather_data)")
print("  • get_weather_simple(location, target_datetime=None)")
print("\n🎯 Output Categories: Clear | Cloudy | Rainy | Foggy")
print("\n💡 Usage:")
print("-" * 70)
print("\n1️⃣ Get current weather with category:")
print("   weather = get_weather_simple('Bhopal')")
print("\n2️⃣ Get forecast with category:")
print("   forecast = get_weather_simple('Delhi', '2026-04-20 15:00')")
print("\n3️⃣ Add category to existing data:")
print("   weather = get_current_weather('Mumbai')")
print("   weather_with_cat = process_weather_with_category(weather)")
print("\n" + "="*70)

In [0]:
# ============================================================================
# WEATHER DATA FRAMEWORK - SIMPLE INTERFACE
# ============================================================================

def get_weather(location, date=None, time=None):
    """
    🌤️ GET WEATHER DATA - Simplified Interface
    
    INPUT REQUIREMENTS:
    -------------------
    1. location (REQUIRED) - City name as string
       Example: "Bhopal", "Delhi", "Mumbai"
    
    2. date (OPTIONAL) - Date string
       Format: "YYYY-MM-DD" or "DD-MM-YYYY"
       If None: Returns CURRENT weather
       If provided: Returns FORECAST for that date
    
    3. time (OPTIONAL) - Time string
       Format: "HH:MM" (24-hour format)
       Default: "00:00" if not provided
       Only used when date is provided
    
    OUTPUT:
    -------
    Dictionary with 4 fields:
    {
        'location': str,              # City name
        'timestamp': str,             # "YYYY-MM-DD HH:MM"
        'temperature_celsius': float, # Temperature in °C
        'weather_category': str       # "Clear" / "Cloudy" / "Rainy" / "Foggy"
    }
    """
    
    # Validate location
    if not location or not isinstance(location, str):
        print("❌ ERROR: 'location' is required and must be a string")
        return None
    
    # If no date provided, return current weather
    if date is None:
        print(f"📍 Fetching CURRENT weather for: {location}")
        result = get_weather_simple(location)
        return result
    
    # Parse date format
    try:
        if '-' in date:
            parts = date.split('-')
            if len(parts[0]) == 4:  # YYYY-MM-DD
                formatted_date = date
            else:  # DD-MM-YYYY
                formatted_date = f"{parts[2]}-{parts[1]}-{parts[0]}"
        else:
            print("❌ ERROR: Date must be in format 'YYYY-MM-DD' or 'DD-MM-YYYY'")
            return None
        datetime.strptime(formatted_date, "%Y-%m-%d")
    except ValueError:
        print(f"❌ ERROR: Invalid date format. Use 'YYYY-MM-DD' or 'DD-MM-YYYY'")
        return None
    
    # Parse time
    if time is None:
        time = "00:00"
    try:
        datetime.strptime(time, "%H:%M")
    except ValueError:
        print("❌ ERROR: Invalid time format. Use 'HH:MM' (24-hour format)")
        return None
    
    # Combine date and time
    target_datetime = f"{formatted_date} {time}"
    print(f"📍 Fetching FORECAST for: {location} at {target_datetime}")
    
    result = get_weather_simple(location, target_datetime)
    if isinstance(result, list) and len(result) > 0:
        result = result[0]
    return result


def get_weather_bulk(locations, date=None, time=None):
    """
    Get weather for multiple locations at once
    
    Parameters:
    -----------
    locations : list of str
        List of city names
    date, time : optional date and time
    
    Returns: Spark DataFrame
    """
    if not locations or not isinstance(locations, list):
        print("❌ ERROR: 'locations' must be a list of city names")
        return None
    
    print(f"🌐 Fetching weather for {len(locations)} locations...")
    results = []
    for location in locations:
        weather = get_weather(location, date, time)
        if weather:
            results.append(weather)
    
    if results:
        df = weather_to_dataframe(results)
        return df
    return None


print("="*70)
print("🌤️  WEATHER FRAMEWORK - REQUIREMENTS")
print("="*70)

print("\n📋 WHAT YOU NEED TO PROVIDE:")
print("-" * 70)

print("\n1️⃣  LOCATION (Required)")
print("    • Type: String")
print("    • Example: 'Bhopal', 'Delhi', 'Mumbai'")
print("    • What it is: City or station name")

print("\n2️⃣  DATE (Optional)")
print("    • Type: String")
print("    • Format: 'YYYY-MM-DD' or 'DD-MM-YYYY'")
print("    • Example: '2026-04-22' or '22-04-2026'")
print("    • If NOT provided → Returns CURRENT weather")
print("    • If provided → Returns FORECAST for that date")

print("\n3️⃣  TIME (Optional)")
print("    • Type: String")
print("    • Format: 'HH:MM' (24-hour)")
print("    • Example: '14:30', '09:00', '23:45'")
print("    • Default: '00:00' (midnight)")
print("    • Only matters when DATE is provided")

print("\n\n📤 WHAT YOU GET BACK:")
print("-" * 70)
print("\n4 pieces of information:")
print("  ✓ location            - City name")
print("  ✓ timestamp           - Date & time (YYYY-MM-DD HH:MM)")
print("  ✓ temperature_celsius - Temperature in °C")
print("  ✓ weather_category    - ONE OF: 'Clear' / 'Cloudy' / 'Rainy' / 'Foggy'")

print("\n\n💡 HOW TO USE:")
print("-" * 70)

print("\n# Get CURRENT weather")
print("weather = get_weather('Bhopal')")

print("\n# Get FORECAST for specific date (midnight)")
print("weather = get_weather('Delhi', date='2026-04-22')")

print("\n# Get FORECAST for specific date & time")
print("weather = get_weather('Mumbai', date='2026-04-25', time='15:30')")

print("\n# Get weather for MULTIPLE locations")
print("df = get_weather_bulk(['Bhopal', 'Delhi', 'Mumbai'], date='2026-04-22')")

print("\n\n🎯 WEATHER CATEGORIES EXPLAINED:")
print("-" * 70)
print("  • Clear  → Clear sky, good visibility, minimal clouds")
print("  • Cloudy → Overcast, heavy clouds (no precipitation)")
print("  • Rainy  → Rain, drizzle, thunderstorms")
print("  • Foggy  → Fog, mist, poor visibility")

print("\n" + "="*70)
print("✅ Framework ready! Tell me location, date, and time.")
print("="*70)

In [0]:
# ============================================================================
# DEMO: 4-CATEGORY WEATHER CLASSIFICATION
# ============================================================================

print("🌤️ DEMO: SIMPLIFIED WEATHER CATEGORIES")
print("="*70)
print("\n🎯 Output: 4 categories - Clear, Cloudy, Rainy, Foggy\n")

# ============================================================================
# Test 1: Current Weather for Multiple Cities
# ============================================================================

print("\n1️⃣ CURRENT WEATHER - MULTIPLE CITIES")
print("-" * 70)

cities = ['Bhopal', 'Delhi', 'Mumbai', 'Bangalore', 'Kolkata']
results = []

for city in cities:
    weather = get_weather_simple(city)
    if weather:
        results.append(weather)
        print(f"\n📍 {weather['location']:<15} | {weather['timestamp']:<20} | {weather['temperature_celsius']}°C | 🌤️ {weather['weather_category']}")

print("\n" + "="*70)

# Create DataFrame
if results:
    df_simple = weather_to_dataframe(results)
    print("\n📊 Simple Weather DataFrame:")
    display(df_simple)


# ============================================================================
# Test 2: Forecast with Categories
# ============================================================================

print("\n\n2️⃣ FORECAST WITH CATEGORIES - NEXT 24 HOURS")
print("-" * 70)

location = "Delhi"
print(f"\n📍 Location: {location}")
print("\n⏰ Next 24 hours forecast:")

# Get full forecast
forecast_full = get_weather_forecast(location)
if forecast_full:
    # Process with categories
    forecast_categorized = process_weather_with_category(forecast_full)
    
    # Take first 24 hours
    forecast_24h = forecast_categorized[:24]
    
    # Simplify output
    simple_forecast = []
    for f in forecast_24h:
        simple_record = {
            'location': f['location'],
            'timestamp': f['forecast_time'],
            'temperature_celsius': f['temperature_celsius'],
            'weather_category': f['weather_category']
        }
        simple_forecast.append(simple_record)
    
    # Show first 6 records
    print("\n🕰️ First 6 hours:")
    for i, record in enumerate(simple_forecast[:6]):
        print(f"  {i+1}. {record['timestamp']:<20} | {record['temperature_celsius']}°C | 🌤️ {record['weather_category']}")
    
    # Create DataFrame
    df_forecast_simple = weather_to_dataframe(simple_forecast)
    print("\n📊 Forecast DataFrame (24 hours):")
    display(df_forecast_simple)
    
    # Count by category
    print("\n📈 Weather Category Distribution (24 hours):")
    df_forecast_simple.groupBy('weather_category').count().orderBy('count', ascending=False).show()


# ============================================================================
# Test 3: Specific Date/Time Query
# ============================================================================

print("\n\n3️⃣ SPECIFIC DATE/TIME QUERY")
print("-" * 70)

target_city = "Mumbai"
target_time = "2026-04-20 18:00"

print(f"\n📍 Location: {target_city}")
print(f"⏰ Target: {target_time}")

weather_target = get_weather_simple(target_city, target_time)

if weather_target:
    if isinstance(weather_target, list):
        weather_target = weather_target[0]  # Take first match
    
    print(f"\n✅ RESULT:")
    print("="*70)
    print(f"📍 Location:    {weather_target['location']}")
    print(f"🕒 Time:        {weather_target['timestamp']}")
    print(f"🌡️ Temperature: {weather_target['temperature_celsius']}°C")
    print(f"🌤️ Weather:     {weather_target['weather_category']}")
    print("="*70)


# ============================================================================
# Summary
# ============================================================================

print("\n\n" + "="*70)
print("✅ CLASSIFICATION COMPLETE!")
print("="*70)
print("\n🎯 Weather Categories Used:")
print("  • Clear  - Clear sky, minimal clouds (<30%)")
print("  • Cloudy - Overcast, heavy clouds (no rain)")
print("  • Rainy  - Rain, drizzle, thunderstorms")
print("  • Foggy  - Fog or mist conditions")
print("\n💡 Integration Ready:")
print("  Use get_weather_simple(location, datetime) for your train routing!")
print("="*70)